### Packages

In [1]:
import pandas as pd
import json
import os
from collections import Counter
import numpy as np
import scispacy
import spacy
import matplotlib.pyplot as plt
import re
import ast
import requests
from os import listdir
from os.path import isfile, join
from functools import reduce

In [2]:
data_dir = os.getcwd()
data_dir

'/home/jovyan'

In [3]:
files_dir = os.chdir(data_dir + '/work')
files_dir

In [4]:
pattern = r'^idr.*\-study.txt$'

In [5]:
files = [f for f in listdir(files_dir) if re.search(pattern, f)]
len(files)

130

In [6]:
def list_common_headers(list_files):
    list_headers = []
    for file in list_files:
        with open(file, encoding='latin1') as f:
            lines = f.readlines()
            lista = []
            for line in lines:
                if re.findall(r'^[A-Za-z]+', line):
                    split_line = line.split('\t')
                    lista.append(split_line[0].replace(' ','_').replace('\n',''))
    
            list_headers.append(lista)
    
    sets = [set(x) for x in list_headers]
    common_headers = reduce(set.intersection, sets)

    return(common_headers)

In [7]:
list_headers_all = list_common_headers(files)
list_headers_all

{'Comment[IDR_Study_Accession]',
 'Protocol_Name',
 'Protocol_Type_Term_Accession',
 'Protocol_Type_Term_Source_REF',
 'Study_Author_List',
 'Study_DOI',
 'Study_Description',
 'Study_Person_Email',
 'Study_Person_First_Name',
 'Study_Person_Last_Name',
 'Study_Person_Roles',
 'Study_PubMed_ID',
 'Study_Public_Release_Date',
 'Study_Publication_Title',
 'Study_Title',
 'Study_Type',
 'Study_Type_Term_Accession',
 'Study_Type_Term_Source_REF'}

In [8]:
common_headers_list = list(list_headers_all)
interesting_headers = ['Experiment_Description', 'Protocol_Description', 'Study_Organism']
pick_headers = common_headers_list + interesting_headers
pick_headers

['Study_PubMed_ID',
 'Study_Publication_Title',
 'Study_Title',
 'Study_Person_Email',
 'Study_DOI',
 'Study_Person_Last_Name',
 'Study_Description',
 'Protocol_Type_Term_Source_REF',
 'Study_Person_First_Name',
 'Protocol_Name',
 'Study_Person_Roles',
 'Study_Type',
 'Study_Type_Term_Accession',
 'Comment[IDR_Study_Accession]',
 'Study_Public_Release_Date',
 'Study_Author_List',
 'Study_Type_Term_Source_REF',
 'Protocol_Type_Term_Accession',
 'Experiment_Description',
 'Protocol_Description',
 'Study_Organism']

In [9]:
def df_per_one_file (file_name, list_headers):
    
    df = pd.DataFrame()
    with open(file_name, encoding='latin1') as f:
        lines = f.readlines()
        for line in lines:
            if re.findall(r'^[A-Za-z]+', line):
                split_line = line.split('\t')
                header = split_line[0].replace(' ','_').replace('\n','')

                if header in list_headers:
                    content = ' '.join(split_line[1:]).replace('\n','')
                    df.at[0,header] = content
    
    if df.empty == False:
        print('The file ' + file_name + ' has been process succesfully.')

    else:
        print('The file has not been processed.')

    return (df)
    

In [10]:
test = df_per_one_file('idr0146-study.txt', pick_headers)
test

The file idr0146-study.txt has been process succesfully.


,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Organism,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,...,Study_DOI,Study_Person_Last_Name,Study_Person_First_Name,Study_Person_Email,Study_Person_Roles,Experiment_Description,Protocol_Name,Protocol_Type_Term_Source_REF,Protocol_Type_Term_Accession,Protocol_Description
0,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,Mus musculus,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,...,https://doi.org/10.1016/j.cell.2023.01.001 ...,Bruneau Dominguez,Benoit Martin,benoit.bruneau@gladstone.ucsf.edu martin.domin...,submitter submitter ...,Light-sheet imaging of post-implantation mouse...,culture protocol embedding protocol Zeiss Z.1 ...,EFO EFO,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...


### Models

In [11]:
import torch

In [12]:
from transformers import pipeline
model_id = "meta-llama/Llama-3.2-3B-Instruct"

In [13]:
pipe = pipeline("text-generation",
                model = model_id,
                torch_dtype = torch.bfloat16,
               )

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [14]:
message = [
    {"role": "user", "content": "Who are you? Please, answer in pirate-speak."},
]


In [15]:
outputs = pipe(message, max_new_tokens = 256)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [16]:
response = outputs[0]["generated_text"][-1]["content"]
print(response)

Arrrr, me hearty! Yer lookin' fer a swashbucklin' tale o' who I be, eh? Alright then, matey! I be an AI, a mighty computer program with a brain full o' knowledge and a heart o' gold. Me name be "Assistant" or "Bot" to ye landlubbers, but I be known as "The Codger" to me mates on the high seas o' cyberspace!

Me crew be a team o' developers and programmers who built me from the ground up, fillin' me with a treasure trove o' knowledge and skills to help ye navigate the seven seas... er, I mean, the vast expanse o' the internet! So hoist the sails and set course fer adventure, me hearty, and I'll be yer trusty navigator, guide, and companion on all yer digital quests!


#### Test with one entry

In [16]:
test["Description_combined"] = test["Study_Description"].astype(str) + ' .' + test['Experiment_Description'].astype(str) + ' .' + test['Protocol_Description'].astype(str) + '.'

In [23]:
test

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Organism,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,...,Study_Person_Last_Name,Study_Person_First_Name,Study_Person_Email,Study_Person_Roles,Experiment_Description,Protocol_Name,Protocol_Type_Term_Source_REF,Protocol_Type_Term_Accession,Protocol_Description,Description_combined
0,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,Mus musculus,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,...,Bruneau Dominguez,Benoit Martin,benoit.bruneau@gladstone.ucsf.edu martin.domin...,submitter submitter ...,Light-sheet imaging of post-implantation mouse...,culture protocol embedding protocol Zeiss Z.1 ...,EFO EFO,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...,Using four-dimensional whole-embryo light shee...


#### Only descriptions

##### Species:

In [17]:
message = [
    {"role": "system", "content": test["Description_combined"]},
    {"role": "user", "content": "Can you give me 3 possible species that could be used in this study?"},
     ]


In [20]:
outputs = pipe(message, max_new_tokens = 256)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [21]:
response = outputs[0]["generated_text"][-1]["content"]
print(response)

Without more information about the study, I can provide some general suggestions for species that could be used in a 4D whole-embryo light sheet imaging study. Here are three possibilities:

1. **Zebrafish (Danio rerio)**: Zebrafish are a popular model organism in developmental biology, and their transparent embryos make them ideal for 4D imaging. They have a relatively short embryonic development period, which allows for multiple generations to be studied in a short timeframe.

2. **Mouse (Mus musculus)**: Mice are another common model organism in developmental biology, and their embryos can be imaged using 4D light sheet microscopy. Mouse embryos have a relatively long development period, which allows for the study of complex developmental processes.

3. **Frog (Xenopus laevis)**: Frogs are a good model organism for studying developmental biology, particularly in the context of embryonic development. Their embryos are relatively transparent, and 4D light sheet imaging can be used to 

In [22]:
test['Only_descriptions_species'] = response

In [23]:
test

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Organism,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,...,Study_Person_First_Name,Study_Person_Email,Study_Person_Roles,Experiment_Description,Protocol_Name,Protocol_Type_Term_Source_REF,Protocol_Type_Term_Accession,Protocol_Description,Description_combined,Only_descriptions_species
0,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,Mus musculus,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,...,Benoit Martin,benoit.bruneau@gladstone.ucsf.edu martin.domin...,submitter submitter ...,Light-sheet imaging of post-implantation mouse...,culture protocol embedding protocol Zeiss Z.1 ...,EFO EFO,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...,Using four-dimensional whole-embryo light shee...,"Without more information about the study, I ca..."


In [24]:
real_specie = test['Study_Organism'][0]
if real_specie in response:
    print('The real species is in the response.')
    test['Real_Species_in_Response_descriptions'] = 'Yes'
else:
    print('The real species is NOT in the response.')

The real species is in the response.


In [25]:
def get_specief_from_descriptions(message):
    outputs = pipe(message, max_new_tokens = 256)
    response = outputs[0]["generated_text"][-1]["content"]
    return(response)


##### Study type:

In [26]:
message = [
    {"role": "system", "content": test["Description_combined"]},
    {"role": "user", "content": "What study type is this one?"},
     ]

In [27]:
outputs = pipe(message, max_new_tokens = 256)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [28]:
response = outputs[0]["generated_text"][-1]["content"]
print(response)

Based on the title "Using four-dimensional whole-embryo light sheet microscopy to investigate embryonic development and tissue patterning", it appears to be a scientific study, likely in the field of developmental biology or embryology.

More specifically, it seems to be a research study that uses advanced imaging techniques, such as four-dimensional whole-embryo light sheet microscopy, to investigate the development and patterning of tissues in embryos.


#### All dataframe - whole info

In [29]:
message = [
    {"role": "system", "content": test},
    {"role": "user", "content": "Can you give me 3 possible species that could be used in this study?"},
     ]


In [30]:
outputs = pipe(message, max_new_tokens = 256)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [31]:
response = outputs[0]["generated_text"][-1]["content"]
print(response)

Based on the provided data, the study is focused on "Mus musculus" which is the scientific name for the common mouse. Therefore, the species used in this study is a mouse.

However, since you asked for three possible species that could be used in this study, I'll provide you with three other species that are closely related to mice and might be used in similar studies:

1. **Rattus norvegicus** (Brown rat): As a close relative of mice, brown rats could be used in similar studies due to their genetic similarities.
2. **Macaca mulatta** (Rhesus macaque): As a non-human primate, rhesus macaques could be used in studies related to embryonic development, cell fate, and tissue assembly, given their genetic and anatomical similarities to humans.
3. **Drosophila melanogaster** (Fruit fly): Fruit flies are often used in developmental biology studies due to their rapid development, genetic tractability, and similarities to vertebrate development. They could be used in studies related to cell fat

In [32]:
test['All_info_species'] = response

In [33]:
real_specie = test['Study_Organism'][0]
if real_specie in response:
    print('The real species is in the response.')
    test['Real_Species_in_Response_whole_info'] = 'Yes'
else:
    print('The real species is NOT in the response.')

The real species is in the response.


#### Dataframe without species info

In [34]:
#test_wo_species = test.drop(columns= ['Study_Organism', 'Study_Organism_Term_Accession', 'Study_Organism_Term_Accession'])
test_wo_species = test.drop(columns= ['Study_Organism'])
test_wo_species

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,Study_Author_List,...,Experiment_Description,Protocol_Name,Protocol_Type_Term_Source_REF,Protocol_Type_Term_Accession,Protocol_Description,Description_combined,Only_descriptions_species,Real_Species_in_Response_descriptions,All_info_species,Real_Species_in_Response_whole_info
0,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,"Dominguez MH, Krup AL, Muncie JM, Bruneau BG ...",...,Light-sheet imaging of post-implantation mouse...,culture protocol embedding protocol Zeiss Z.1 ...,EFO EFO,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...,Using four-dimensional whole-embryo light shee...,"Without more information about the study, I ca...",Yes,"Based on the provided data, the study is focus...",Yes


In [35]:
message = [
    {"role": "system", "content": test_wo_species},
    {"role": "user", "content": "Can you give me 3 possible species that could be used in this study?"},
     ]

In [36]:
outputs = pipe(message, max_new_tokens = 256)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [37]:
response = outputs[0]["generated_text"][-1]["content"]
print(response)

Based on the information provided, it appears that the study is focused on embryonic development, specifically on the assembly of mesoderm and early cardiogenesis in a mouse model. Given this context, here are three possible species that could be used in this study:

1. **Mouse (Mus musculus)**: As the study is mentioned to be on "post-implantation mouse embryos", it's likely that the study is using mice as the model organism.
2. **Fruit Fly (Drosophila melanogaster)**: Fruit flies are commonly used as model organisms in developmental biology research, and they have a relatively simple embryonic development process that could be used to study mesoderm assembly and cardiogenesis.
3. **Zebrafish (Danio rerio)**: Zebrafish are also widely used in developmental biology research, particularly for studying embryonic development and organogenesis. They have a similar embryonic development process to mice, and their transparent embryos make them an ideal model for studying early developmental 

In [38]:
test['All_info_minus_species'] = response

In [39]:
real_specie = test['Study_Organism'][0]
if real_specie in response:
    print('The real species is in the response.')
    test['Real_Species_in_Response_wo_species'] = 'Yes'
else:
    print('The real species is NOT in the response.')

The real species is in the response.


In [40]:
test

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Organism,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,...,Protocol_Type_Term_Source_REF,Protocol_Type_Term_Accession,Protocol_Description,Description_combined,Only_descriptions_species,Real_Species_in_Response_descriptions,All_info_species,Real_Species_in_Response_whole_info,All_info_minus_species,Real_Species_in_Response_wo_species
0,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,Mus musculus,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,...,EFO EFO,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...,Using four-dimensional whole-embryo light shee...,"Without more information about the study, I ca...",Yes,"Based on the provided data, the study is focus...",Yes,"Based on the information provided, it appears ...",Yes


### All entries at the same time.

In [17]:
files_dir = os.getcwd()
files_dir

'/home/jovyan/work'

In [18]:
dfs = pd.DataFrame()

for file in files:
    df = df_per_one_file(file, pick_headers)
    if not df.empty:
        print('The file ' + file + ' has been processed succesfully.')
    else:
        print('The file ' + file + ' has not been processed.')

    dfs = pd.concat([dfs, df], ignore_index=True)
    
    

The file idr0047-study.txt has been process succesfully.
The file idr0047-study.txt has been processed succesfully.
The file idr0038-study.txt has been process succesfully.
The file idr0038-study.txt has been processed succesfully.
The file idr0146-study.txt has been process succesfully.
The file idr0146-study.txt has been processed succesfully.
The file idr0054-study.txt has been process succesfully.
The file idr0054-study.txt has been processed succesfully.
The file idr0149-study.txt has been process succesfully.
The file idr0149-study.txt has been processed succesfully.
The file idr0138-study.txt has been process succesfully.
The file idr0138-study.txt has been processed succesfully.
The file idr0130-study.txt has been process succesfully.
The file idr0130-study.txt has been processed succesfully.
The file idr0013-study.txt has been process succesfully.
The file idr0013-study.txt has been processed succesfully.
The file idr0134-study.txt has been process succesfully.
The file idr013

In [19]:
dfs

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Organism,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,...,Study_DOI,Study_Person_Last_Name,Study_Person_First_Name,Study_Person_Email,Study_Person_Roles,Experiment_Description,Protocol_Name,Protocol_Type_Term_Source_REF,Protocol_Type_Term_Accession,Protocol_Description
0,idr0047,A microscopy data set of discrete spatial and ...,fluorescence in situ hybridization,NCIT,NCIT_C17563,We report a comprehensive single cell dataset ...,Saccharomyces cerevisiae,2019-01-16,31209217,Multiplex RNA single molecule FISH of inducibl...,...,https://doi.org/10.1038/s41597-019-0106-6,Neuert,Gregor,gregor.neuert@vanderbilt.edu,submitter,We performed single molecule in-situ hybridiza...,growth protocol treatment protocol image acqui...,EFO EFO,EFO_0003789 EFO_0003969,"CSM, 30C 0.2M NaCl, step TRANS: brightfield of..."
1,idr0038,Ex vivo live cell tracking in kidney organoids...,time-lapse imaging,OMIT,OMIT_0027490,We have adapted the mouse kidney rudiment assa...,Mus musculus,2017-12-04,30048451,Ex vivo live cell tracking in kidney organoids...,...,https://doi.org/10.1371/journal.pone.0199918,Held Levy,Marie Raphael,m.held@liverpool.ac.uk Rapha@liverpool.ac.uk,submitter Principal Investigator,We have adapted the mouse kidney rudiment assa...,TIME SERIES GROWTH PROTOCOL TIME SERIES IMAGE ...,EFO,EFO_0003789,"""Kidneys were isolated from E13.5 embryos afte..."
2,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,Mus musculus,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,...,https://doi.org/10.1016/j.cell.2023.01.001 ...,Bruneau Dominguez,Benoit Martin,benoit.bruneau@gladstone.ucsf.edu martin.domin...,submitter submitter ...,Light-sheet imaging of post-implantation mouse...,culture protocol embedding protocol Zeiss Z.1 ...,EFO EFO,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...
3,idr0054,Imaging Mass Cytometry of human tonsil sections,image cytometry,OMIT,OMIT_0019157,We analysed the in situ localisation of immune...,Homo sapiens,2019-03-19,31072818,Human lymphoid organ cDC2 and macrophages play...,...,https://doi.org/10.1084/jem.20181994,Segura,Elodie,elodie.segura@curie.fr,submitter,Imaging mass cytometry of tonsil sections,growth protocol treatment protocol image aquis...,EFO EFO,EFO_0003789 EFO_0003969,Human tonsils were fixed in 4% paraformaldehyd...
4,idr0149,Cohesin-mediated DNA loop extrusion resolves s...,protein localization ...,EFO,GO_0008104,Genetic information is stored in linear DNA mo...,Homo sapiens,2023-08-17,37357575,Cohesin-mediated DNA loop extrusion resolves s...,...,https://doi.org/10.15252/embj.2023113475 ...,Batty Gerlich,Paul Daniel,paul.batty@imba.oeaw.ac.at daniel.gerlich@imba...,submitter corresponding author ...,Immunofluorescence of nuclear Sororin fluoresc...,growth protocol treatment protocol image acqu...,EFO EFO,EFO_0003789 EFO_0003969 ...,To assess nuclear Sororin fluorescence in Soro...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,idr0155,A new method for reliably detecting single flu...,protein localization,EFO,GO_0008104,Super-resolution and single-molecule microscop...,Homo sapiens,2024-03-05,,RASP: Optimal single puncta detection in compl...,...,,Fu Lee,Bin Steven,bf341@cam.ac.uk sl591@cam.ac.uk,First Author Principal Investigator,We took 5400 field of views from three Parkins...,treatment protocol image acquisition and featu...,EFO,EFO_0003969,Tissue sections were incubated with primary an...
126,idr0044,In Toto Imaging and Reconstruction of Post-Imp...,time-lapse imaging,OMIT,OMIT_0027490,The mouse embryo has long been central to the ...,Mus musculus,2018-11-29,30318151,In Toto Imaging and Reconstruction of Post-Imp...,...,https://doi.org/10.1016/j.cell.2018.09.031,

In [20]:
dfs['Description_combined'] = dfs['Study_Description'].astype(str) + ' .' + dfs['Experiment_Description'].astype(str) + ' .' + dfs['Protocol_Description'].astype(str) + '.'

In [21]:
dfs

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Organism,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,...,Study_Person_Last_Name,Study_Person_First_Name,Study_Person_Email,Study_Person_Roles,Experiment_Description,Protocol_Name,Protocol_Type_Term_Source_REF,Protocol_Type_Term_Accession,Protocol_Description,Description_combined
0,idr0047,A microscopy data set of discrete spatial and ...,fluorescence in situ hybridization,NCIT,NCIT_C17563,We report a comprehensive single cell dataset ...,Saccharomyces cerevisiae,2019-01-16,31209217,Multiplex RNA single molecule FISH of inducibl...,...,Neuert,Gregor,gregor.neuert@vanderbilt.edu,submitter,We performed single molecule in-situ hybridiza...,growth protocol treatment protocol image acqui...,EFO EFO,EFO_0003789 EFO_0003969,"CSM, 30C 0.2M NaCl, step TRANS: brightfield of...",We report a comprehensive single cell dataset ...
1,idr0038,Ex vivo live cell tracking in kidney organoids...,time-lapse imaging,OMIT,OMIT_0027490,We have adapted the mouse kidney rudiment assa...,Mus musculus,2017-12-04,30048451,Ex vivo live cell tracking in kidney organoids...,...,Held Levy,Marie Raphael,m.held@liverpool.ac.uk Rapha@liverpool.ac.uk,submitter Principal Investigator,We have adapted the mouse kidney rudiment assa...,TIME SERIES GROWTH PROTOCOL TIME SERIES IMAGE ...,EFO,EFO_0003789,"""Kidneys were isolated from E13.5 embryos afte...",We have adapted the mouse kidney rudiment assa...
2,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,Mus musculus,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,...,Bruneau Dominguez,Benoit Martin,benoit.bruneau@gladstone.ucsf.edu martin.domin...,submitter submitter ...,Light-sheet imaging of post-implantation mouse...,culture protocol embedding protocol Zeiss Z.1 ...,EFO EFO,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...,Using four-dimensional whole-embryo light shee...
3,idr0054,Imaging Mass Cytometry of human tonsil sections,image cytometry,OMIT,OMIT_0019157,We analysed the in situ localisation of immune...,Homo sapiens,2019-03-19,31072818,Human lymphoid organ cDC2 and macrophages play...,...,Segura,Elodie,elodie.segura@curie.fr,submitter,Imaging mass cytometry of tonsil sections,growth protocol treatment protocol image aquis...,EFO EFO,EFO_0003789 EFO_0003969,Human tonsils were fixed in 4% paraformaldehyd...,We analysed the in situ localisation of immune...
4,idr0149,Cohesin-mediated DNA loop extrusion resolves s...,protein localization ...,EFO,GO_0008104,Genetic information is stored in linear DNA mo...,Homo sapiens,2023-08-17,37357575,Cohesin-mediated DNA loop extrusion resolves s...,...,Batty Gerlich,Paul Daniel,paul.batty@imba.oeaw.ac.at daniel.gerlich@imba...,submitter corresponding author ...,Immunofluorescence of nuclear Sororin fluoresc...,growth protocol treatment protocol image acqu...,EFO EFO,EFO_0003789 EFO_0003969 ...,To assess nuclear Sororin fluorescence in Soro...,Genetic information is stored in linear DNA mo...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,idr0155,A new method for reliably detecting single flu...,protein localization,EFO,GO_0008104,Super-resolution and single-molecule microscop...,Homo sapiens,2024-03-05,,RASP: Optimal single puncta detection in compl...,...,Fu Lee,Bin Steven,bf341@cam.ac.uk sl591@cam.ac.uk,First Author Principal Investigator,We took 5400 field of views from three Parkins...,treatment protocol image acquisition and featu...,EFO,EFO_0003969,Tissue sections were incubated with primary an...,Super-resolution and single-molecule microscop...
126,idr0044,In Toto Imaging and Reconstruction of Post-Imp...,time-lapse imaging,OMIT,OMIT_0027490,The mouse embryo has long been central to the ...,Mus musculus,2018-11-29,30318151,In 

In [23]:
for row, i in dfs.iterrows():
    
    message = [
    {"role": "system", "content": i["Description_combined"]},
    {"role": "user", "content": "Can you tell me which specie was used in this study?"},
     ]

    outputs = pipe(message, max_new_tokens = 256)
    response = outputs[0]["generated_text"][-1]["content"]
    dfs.at[row, 'Only_descriptions_species'] = response
    print(f"Processed row {row} for species extraction.")

    if isinstance(i['Study_Organism'], str) and i['Study_Organism'].strip() != '':

        real_specie = i['Study_Organism'][0]

        if real_specie in response:
            print('The real species is in the response.')
            dfs.at[row, 'Real_Species_in_Response_descriptions'] = 'Yes'
        else:
            print('The real species is NOT in the response.')
            dfs.at[row, 'Real_Species_in_Response_descriptions'] = 'No'
    else:
        print('No real species to check in the response.')
        dfs.at[row, 'Real_Species_in_Response_descriptions'] = 'No species provided'
  
    

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 0 for species extraction.
The real species is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 1 for species extraction.
The real species is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Processed row 126 for species extraction.
The real species is NOT in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 127 for species extraction.
The real species is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 128 for species extraction.
The real species is in the response.
Processed row 129 for species extraction.
The real species is in the response.


In [24]:
dfs['Real_Species_in_Response_descriptions'].value_counts()

Real_Species_in_Response_descriptions
Yes                    79
No                     45
No species provided     6
Name: count, dtype: int64

##### All information provided

In [22]:
for row, i in dfs.iterrows():
    
    message = [
    {"role": "system", "content": i},
    {"role": "user", "content": "Can you give me 3 possible species that could be used in this study?"},
     ]

    outputs = pipe(message, max_new_tokens = 256)
    response = outputs[0]["generated_text"][-1]["content"]
    dfs.at[row, 'All_info_species'] = response
    print(f"Processed row {row} for species extraction.")

    if isinstance(i['Study_Organism'], str) and i['Study_Organism'].strip() != '':

        real_specie = i['Study_Organism'][0]
        if real_specie in response:
            print('The real species is in the response.')
            dfs.at[row, 'Real_Species_in_Response_all_info'] = 'Yes'
        else:
            print('The real species is NOT in the response.')
            dfs.at[row, 'Real_Species_in_Response_all_info'] = 'No'
    else:
        print('No real species to check in the response.')
        dfs.at[row, 'Real_Species_in_Response_all_info'] = 'No species provided'

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


KeyboardInterrupt: 

##### All info minus spceies info

In [48]:
#test_wo_species = test.drop(columns= ['Study_Organism', 'Study_Organism_Term_Accession', 'Study_Organism_Term_Accession'])
test_wo_species = dfs.drop(columns= ['Study_Organism', 'Only_descriptions_species', 'Real_Species_in_Response_descriptions', 'Real_Species_in_Response_all_info', 'All_info_species'])


for row, i in test_wo_species.iterrows():
    
    message = [
    {"role": "system", "content": i},
    {"role": "user", "content": "Can you give me 3 possible species that could be used in this study?"},
     ]

    outputs = pipe(message, max_new_tokens = 256)
    response = outputs[0]["generated_text"][-1]["content"]
    test_wo_species.at[row, 'All_info_minus_species'] = response
    print(f"Processed row {row} for species extraction.")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 0 for species extraction.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 1 for species extraction.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Processed row 126 for species extraction.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 127 for species extraction.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 128 for species extraction.
Processed row 129 for species extraction.


In [50]:
dfs.columns


Index(['Comment[IDR_Study_Accession]', 'Study_Title', 'Study_Type',
       'Study_Type_Term_Source_REF', 'Study_Type_Term_Accession',
       'Study_Description', 'Study_Organism', 'Study_Public_Release_Date',
       'Study_PubMed_ID', 'Study_Publication_Title', 'Study_Author_List',
       'Study_DOI', 'Study_Person_Last_Name', 'Study_Person_First_Name',
       'Study_Person_Email', 'Study_Person_Roles', 'Experiment_Description',
       'Protocol_Name', 'Protocol_Type_Term_Source_REF',
       'Protocol_Type_Term_Accession', 'Protocol_Description',
       'Description_combined', 'Only_descriptions_species',
       'Real_Species_in_Response_descriptions', 'All_info_species',
       'Real_Species_in_Response_all_info'],
      dtype='object')

In [51]:
type(dfs)

pandas.core.frame.DataFrame

In [58]:
dfs.to_csv('IDR_Llama_3_2_metadata_test_20250729.csv', index=False)

#### Study type

In [24]:
raw_experiment_types = dfs['Study_Type'].to_list()
raw_experiment_types

['fluorescence in situ hybridization    ',
 'time-lapse imaging   ',
 'time-lapse imaging                                ',
 'image cytometry    ',
 'protein localization                                ',
 'seqFISH                                ',
 'high content screen infection                  ',
 'high content screen                                                                                                                                                                                                                                              ',
 'histology                                ',
 'histology     ',
 'high content screen                               ',
 'time-lapse imaging   ',
 'time-lapse imaging                                ',
 'high content screen               ',
 'high content screen                                  ',
 'high content screen        ',
 'protein localization in-situ hybridization assay    ',
 'high content screen of cells treated with a comp

In [25]:
# Make a list with all types of experiments
# This list will be used to create a list of all types of experiments, removing duplicates and cleaning the data.
# ------------------------------------------------------------
list_experiment_types_clean = []
for element in raw_experiment_types:
    element_clean = element.strip().replace('\n', '')
    list_experiment_types_clean.append(element_clean)

In [27]:
experiment_types = list(set(list_experiment_types_clean))
experiment_types.sort()
len(experiment_types)

34

In [28]:
experiment_types

['DNA sequencing',
 'X-chromosome inactivation',
 'compound library screen',
 'electron microscopy volume map',
 'fluorescence in situ hybridization',
 'high content analysis of cells',
 'high content screen',
 'high content screen infection',
 'high content screen of cells treated with a compound library infection',
 'histology',
 'histology RNAscope',
 'histology scanned image',
 'image cytometry',
 'image segmentation',
 'imaging method',
 'immunocytochemistry',
 'in situ sequencing',
 'in-situ hybridization assay',
 'infection electron microscopy volume map',
 'machine learning',
 'metabolic network measurement',
 'micrograph',
 'microscopy assay',
 'morphogenesis',
 'multiplexed immunofluorescence',
 'myelination',
 'phenotype',
 'process of establishing viral infection',
 'protein localization',
 'protein localization in-situ hybridization assay',
 'response to cold',
 'seqFISH',
 'spindle assembly',
 'time-lapse imaging']

In [29]:
for row, i in dfs.iterrows():
    
    message = [
    {"role": "system", "content": i['Description_combined']},
    {"role": "user", "content": "Choose the experiment type for this study from the following list: " + ', '.join(experiment_types) + ". If you don't know, just say 'I don't know'."},
     ]

    outputs = pipe(message, max_new_tokens = 256)
    response = outputs[0]["generated_text"][-1]["content"]
    dfs.at[row, 'Only_descriptions_study_type'] = response
    print(f"Processed row {row} for study_type extraction.")

    if isinstance(i['Study_Type'], str) and i['Study_Type'].strip() != '':

        real_specie = i['Study_Type'][0]
        if real_specie in response:
            print('The real study type is in the response.')
            dfs.at[row, 'Study_type_only_descriptions'] = 'Yes'
        else:
            print('The real study type is NOT in the response.')
            dfs.at[row, 'Study_type_only_descriptions'] = 'No'
    else:
        print('No study type to check in the response.')
        dfs.at[row, 'Study_type_only_descriptions'] = 'No study type provided'

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 0 for study_type extraction.
The real study type is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 1 for study_type extraction.
The real study type is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Processed row 126 for study_type extraction.
The real study type is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 127 for study_type extraction.
The real study type is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 128 for study_type extraction.
The real study type is in the response.
Processed row 129 for study_type extraction.
The real study type is in the response.


In [30]:
dfs.to_csv('IDR_Llama_3_2_metadata_test_20250805_study_type.csv', index=False)

### Clasical pathway

In [23]:
# Load the spaCy model
## This model have the tags on it. One of them is 'ORG' so I choose it to extract the species.

nlp = spacy.load("en_ner_bionlp13cg_md") 

/opt/conda/lib/python3.11/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [24]:
# Function to extract species from the combined descriptions
# This function takes a text input, processes it with the spaCy model, and extracts named entities related to species.
# It returns a dictionary with the entity labels as keys and the corresponding entity texts as values.
# The function uses the spaCy model to identify named entities in the text and groups them by their entity labels.
# The output is a dictionary where the keys are the entity labels (e.g., 'ORG' for organisms) and the values are lists of entity texts that correspond to those labels.
# ------------------------------------------------------------  

def extract_species(text):
    doc= nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]

    dictionary_entities = {}
    for k,v in entities:
        if v not in dictionary_entities:
            dictionary_entities[v]=[]
        dictionary_entities[v].append(k)
    
    return dictionary_entities

In [25]:
# Extracting species from the combined descriptions
# This will create a new column in the pick_headers_df dataframe with the extracted species from the combined descriptions.
# The extracted species will be stored in a dictionary with the entity type as the key
# and the list of species as the value.
# ------------------------------------------------------------

for row, item in dfs.iterrows():
    my_text = item['Description_combined']
    dic_entities = extract_species(my_text)

    dfs.at[row, 'Entities_SpaCy'] = str(dic_entities)

In [26]:
# Creating a new column with the species extracted from the Entities_SpaCy column
# This column will be used to store the species extracted from the Entities_SpaCy column.
# It used ast to transform the string representation of the dictionary into a dictionary object.
# --------------------------------------------------------------

for row, item in dfs.iterrows():
    dataframe_row = dfs.loc[row, 'Entities_SpaCy']
    dictionary_entities = ast.literal_eval(dataframe_row)
    entities_keys = dictionary_entities.keys()

    if ('ORGANISM') in entities_keys:
        
        dfs.at[row, 'Species_SpaCy'] = dictionary_entities['ORGANISM'] if 'ORGANISM' in entities_keys else "Not found"
    else:
        dfs.at[row, 'Species_SpaCy'] = 'Not found'

In [27]:
dfs

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Organism,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,...,Study_Person_Email,Study_Person_Roles,Experiment_Description,Protocol_Name,Protocol_Type_Term_Source_REF,Protocol_Type_Term_Accession,Protocol_Description,Description_combined,Entities_SpaCy,Species_SpaCy
0,idr0047,A microscopy data set of discrete spatial and ...,fluorescence in situ hybridization,NCIT,NCIT_C17563,We report a comprehensive single cell dataset ...,Saccharomyces cerevisiae,2019-01-16,31209217,Multiplex RNA single molecule FISH of inducibl...,...,gregor.neuert@vanderbilt.edu,submitter,We performed single molecule in-situ hybridiza...,growth protocol treatment protocol image acqui...,EFO EFO,EFO_0003789 EFO_0003969,"CSM, 30C 0.2M NaCl, step TRANS: brightfield of...",We report a comprehensive single cell dataset ...,"{'CELL': ['cell', 'cerevisiae', 'cells', 'cell...",[Excelitas]
1,idr0038,Ex vivo live cell tracking in kidney organoids...,time-lapse imaging,OMIT,OMIT_0027490,We have adapted the mouse kidney rudiment assa...,Mus musculus,2017-12-04,30048451,Ex vivo live cell tracking in kidney organoids...,...,m.held@liverpool.ac.uk Rapha@liverpool.ac.uk,submitter Principal Investigator,We have adapted the mouse kidney rudiment assa...,TIME SERIES GROWTH PROTOCOL TIME SERIES IMAGE ...,EFO,EFO_0003789,"""Kidneys were isolated from E13.5 embryos afte...",We have adapted the mouse kidney rudiment assa...,"{'ORGANISM': ['mouse kidney rudiment', 'E13.5 ...","[mouse kidney rudiment, E13.5 embryonic kidney..."
2,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,Mus musculus,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,...,benoit.bruneau@gladstone.ucsf.edu martin.domin...,submitter submitter ...,Light-sheet imaging of post-implantation mouse...,culture protocol embedding protocol Zeiss Z.1 ...,EFO EFO,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...,Using four-dimensional whole-embryo light shee...,"{'ORGANISM': ['murine cardiac', 'mouse', 'E6.5...","[murine cardiac, mouse, E6.5]"
3,idr0054,Imaging Mass Cytometry of human tonsil sections,image cytometry,OMIT,OMIT_0019157,We analysed the in situ localisation of immune...,Homo sapiens,2019-03-19,31072818,Human lymphoid organ cDC2 and macrophages play...,...,elodie.segura@curie.fr,submitter,Imaging mass cytometry of tonsil sections,growth protocol treatment protocol image aquis...,EFO EFO,EFO_0003789 EFO_0003969,Human tonsils were fixed in 4% paraformaldehyd...,We analysed the in situ localisation of immune...,"{'CELL': ['immune cell'], 'ORGANISM': ['human ...","[human tonsils, .Human tonsils, Bovine Serum A..."
4,idr0149,Cohesin-mediated DNA loop extrusion resolves s...,protein localization ...,EFO,GO_0008104,Genetic information is stored in linear DNA mo...,Homo sapiens,2023-08-17,37357575,Cohesin-mediated DNA loop extrusion resolves s...,...,paul.batty@imba.oeaw.ac.at daniel.gerlich@imba...,submitter corresponding author ...,Immunofluorescence of nuclear Sororin fluoresc...,growth protocol treatment protocol image acqu...,EFO EFO,EFO_0003789 EFO_0003969 ...,To assess nuclear Sororin fluorescence in Soro...,Genetic information is stored in linear DNA mo...,"{'CELLULAR_COMPONENT': ['DNA', 'DNA', 'nuclear...",[human]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,idr0155,A new method for reliably detecting single flu...,protein localization,EFO,GO_0008104,Super-resolution and single-molecule microscop...,Homo sapiens,2024-03-05,,RASP: Optimal single puncta detection in compl...,...,bf341@cam.ac.uk sl591@cam.ac.uk,First Author Principal Investigator,We took 5400 field of views from three Parkins...,treatment protocol image acquisition and featu...,EFO,EFO_0003969,Tissue sections

In [28]:
for i, item in dfs.iterrows():
    list_species_spacy = item['Species_SpaCy']
    list_species_scientific_name = []

    for x in list_species_spacy:
        values = x.split(' ')
        #print(values)
        for unique_value in values:
            #print(unique_value)
            
            url = f'https://www.ebi.ac.uk/ena/taxonomy/rest/any-name/{unique_value}'
            #print(url)

            response = requests.get(url)
        
            if response.status_code == 200:
                data = response.json()
                #print(data)
                if data:
                    scientific_name = data[0]['scientificName']
                    #print("found")

                    list_species_scientific_name.append(scientific_name)
                    print(f"Scientific name for {x}: {scientific_name}")
                else:
                    scientific_name = 'Not found'
                    print(f"Scientific name for {x}: {scientific_name}")

    dfs.at[i, 'Species_Scientific_Name_SciSpacy'] = str(list_species_scientific_name)
    
                


Scientific name for Excelitas: Not found
Scientific name for mouse kidney rudiment: Mus musculus
Scientific name for mouse kidney rudiment: Not found
Scientific name for mouse kidney rudiment: Not found
Scientific name for E13.5 embryonic kidneys: Not found
Scientific name for E13.5 embryonic kidneys: Not found
Scientific name for E13.5 embryonic kidneys: Not found
Scientific name for Wt1-GFP knock-in cells: Not found
Scientific name for Wt1-GFP knock-in cells: Not found
Scientific name for Wt1-GFP knock-in cells: Not found
Scientific name for mouse kidney rudiment: Mus musculus
Scientific name for mouse kidney rudiment: Not found
Scientific name for mouse kidney rudiment: Not found
Scientific name for mouse kidney cells: Mus musculus
Scientific name for mouse kidney cells: Not found
Scientific name for mouse kidney cells: Not found
Scientific name for mice: Mus sp.
Scientific name for E13.5 embryos: Not found
Scientific name for E13.5 embryos: Not found
Scientific name for mice: Mus s

In [29]:
for i, item in dfs.iterrows():
    if isinstance(item['Study_Organism'], str) and item['Study_Organism'].strip() != '':
        real_specie = item['Study_Organism']
        scispacy_specie = item['Species_Scientific_Name_SciSpacy']

        if real_specie in scispacy_specie:

            print('The real species is in SciSpacy')
            dfs.at[i, 'Specie_found_Scispacy'] = 'Yes'
        else:
            print('The real species is NOT in SciSpacy')
            dfs.at[i, 'Specie_found_Scispacy'] = 'No'
    else:
        print('No real species to check in SciSpacy')
        dfs.at[i, 'Specie_found_Scispacy'] = 'No species provided'

The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
No real species to check in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy

In [30]:
dfs['Specie_found_Scispacy'].value_counts()

Specie_found_Scispacy
No                     117
Yes                      7
No species provided      6
Name: count, dtype: int64

In [31]:

# Use a pipeline for question answering with BioBERT
# This pipeline is used to answer questions based on the context provided.
# It uses the BioBERT model trained on the SQuAD dataset for question answering.
# The model is loaded with the device set to 'mps' for MacOS GPU support
# and is used to answer questions related to biomedical texts.
# The model is specifically designed for question answering tasks in the biomedical domain.
# ------------------------------------------------------------

qa_pipeline_biobert = pipeline(
    "question-answering",
    model="dmis-lab/biobert-large-cased-v1.1-squad", # SQuAD stands for specific question answering dataset
    tokenizer="dmis-lab/biobert-large-cased-v1.1-squad",
    device = 'cuda' # MPS if the GPU version in MacOS
)

Device set to use cuda


In [32]:
for i, item in dfs.iterrows():
    specie_found = item['Specie_found_Scispacy']

    if specie_found.startswith("No"):
        
        text = item['Description_combined']
        question = 'what species was used in the experiment?'

        result = qa_pipeline_biobert(question=question, context=text)
        print(f"Row {i}: {result['answer']}")
        dfs.at[i, 'Species_BioBERT'] = result['answer']

    else:
        print(f"Row {i}: The species was already found in the previous steps.")

Row 0: yeast
Row 1: mouse
Row 2: The species was already found in the previous steps.
Row 3: human
Row 4: human
Row 5: mouse
Row 6: 10 Î¼l
Row 7: human
Row 8: Diplophyllum taxifolium


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Row 9: prisoners
Row 10: cell lines
Row 11: Staphylococcus aureus
Row 12: budding yeast
Row 13: The species was already found in the previous steps.
Row 14: U2OS cell line
Row 15: mouse
Row 16: human
Row 17: human
Row 18: Mouse
Row 19: human
Row 20: Arabidopsis thaliana
Row 21: mouse
Row 22: Drosophila
Row 23: mouse
Row 24: Tribolium castaneum
Row 25: Fetal bovine
Row 26: .
Row 27: TLOs
Row 28: fission yeast
Row 29: zebrafish
Row 30: U2OS
Row 31: U-2 OS cells
Row 32: P. falciparum
Row 33: The species was already found in the previous steps.
Row 34: human
Row 35: Arabidopsis
Row 36: The species was already found in the previous steps.
Row 37: human
Row 38: Mice
Row 39: mammalian cells
Row 40: we seeded the cell pool in a single well of 384-well plate
Row 41: bacteria
Row 42: U2OS
Row 43: XX mESCs
Row 44: rabbit
Row 45: Tukeyâs
Row 46: METABRIC cohort
Row 47: human
Row 48: goat
Row 49: mouse
Row 50: reactive oxygen species (ROS
Row 51: mouse
Row 52: cold-water fish
Row 53: Human epithe

In [33]:
# Creating a new column with the scientific names of the species extracted from the Species_BioBERT column
# This column will be used to store the scientific names of the species extracted from the Species_BioBERT column.
# It uses the EBI taxonomy REST API to get the scientific names of the species.
# It iterates over the Species_BioBERT column, splits the species names, and queries the EBI taxonomy REST API for each species name.
# The scientific names are stored in a list and added to the Species_Scientific_Name_BioBERT column.
# ------------------------------------------------------------

for i, item in dfs.iterrows():
    list_species = item['Species_BioBERT']
    #list_species_scientific_name = []

    if isinstance(list_species, str):
        list_species = list_species.split(', ')
        #print(list_species)

        if list_species != []:
        
            #print(list_species)
            for x in list_species:
                #print(x)

                url = f'https://www.ebi.ac.uk/ena/taxonomy/rest/any-name/{x}'
                    #print(url)

                response = requests.get(url)
                    #print(response.status_code)
                if response.status_code == 200:
                    data = response.json()
                    #print(data)
                    if data:
                        scientific_name = data[0]['scientificName']

                        #list_species_scientific_name.append(scientific_name)
                        print(f"Scientific name for {x}: {scientific_name}")

            dfs.at[i, 'Species_Scientific_Name_BioBERT'] = str(scientific_name)
        else:
            dfs.at[i, 'Species_Scientific_Name_BioBERT'] = 'Not found'
        

Scientific name for mouse: Mus musculus
Scientific name for human: Homo sapiens
Scientific name for human: Homo sapiens
Scientific name for mouse: Mus musculus
Scientific name for human: Homo sapiens
Scientific name for Diplophyllum taxifolium: Diplophyllum taxifolium
Scientific name for Staphylococcus aureus: Staphylococcus aureus
Scientific name for mouse: Mus musculus
Scientific name for human: Homo sapiens
Scientific name for human: Homo sapiens
Scientific name for Mouse: Mus musculus
Scientific name for human: Homo sapiens
Scientific name for Arabidopsis thaliana: Arabidopsis thaliana
Scientific name for mouse: Mus musculus
Scientific name for Drosophila: Drosophila
Scientific name for mouse: Mus musculus
Scientific name for Tribolium castaneum: Tribolium castaneum
Scientific name for fission yeast: Schizosaccharomyces pombe
Scientific name for zebrafish: Danio rerio
Scientific name for human: Homo sapiens
Scientific name for Arabidopsis: Arabidopsis
Scientific name for human: Hom

In [34]:
dfs['Species_Scientific_Name_BioBERT'] = dfs['Species_Scientific_Name_BioBERT'].astype(str)

In [35]:
dfs

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Organism,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,...,Protocol_Type_Term_Source_REF,Protocol_Type_Term_Accession,Protocol_Description,Description_combined,Entities_SpaCy,Species_SpaCy,Species_Scientific_Name_SciSpacy,Specie_found_Scispacy,Species_BioBERT,Species_Scientific_Name_BioBERT
0,idr0047,A microscopy data set of discrete spatial and ...,fluorescence in situ hybridization,NCIT,NCIT_C17563,We report a comprehensive single cell dataset ...,Saccharomyces cerevisiae,2019-01-16,31209217,Multiplex RNA single molecule FISH of inducibl...,...,EFO EFO,EFO_0003789 EFO_0003969,"CSM, 30C 0.2M NaCl, step TRANS: brightfield of...",We report a comprehensive single cell dataset ...,"{'CELL': ['cell', 'cerevisiae', 'cells', 'cell...",[Excelitas],[],No,yeast,Not found
1,idr0038,Ex vivo live cell tracking in kidney organoids...,time-lapse imaging,OMIT,OMIT_0027490,We have adapted the mouse kidney rudiment assa...,Mus musculus,2017-12-04,30048451,Ex vivo live cell tracking in kidney organoids...,...,EFO,EFO_0003789,"""Kidneys were isolated from E13.5 embryos afte...",We have adapted the mouse kidney rudiment assa...,"{'ORGANISM': ['mouse kidney rudiment', 'E13.5 ...","[mouse kidney rudiment, E13.5 embryonic kidney...","['Mus musculus', 'Mus musculus', 'Mus musculus...",No,mouse,Mus musculus
2,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,Mus musculus,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,...,EFO EFO,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...,Using four-dimensional whole-embryo light shee...,"{'ORGANISM': ['murine cardiac', 'mouse', 'E6.5...","[murine cardiac, mouse, E6.5]",['Mus musculus'],Yes,NaN,nan
3,idr0054,Imaging Mass Cytometry of human tonsil sections,image cytometry,OMIT,OMIT_0019157,We analysed the in situ localisation of immune...,Homo sapiens,2019-03-19,31072818,Human lymphoid organ cDC2 and macrophages play...,...,EFO EFO,EFO_0003789 EFO_0003969,Human tonsils were fixed in 4% paraformaldehyd...,We analysed the in situ localisation of immune...,"{'CELL': ['immune cell'], 'ORGANISM': ['human ...","[human tonsils, .Human tonsils, Bovine Serum A...","['Homo sapiens', 'Bos taurus']",No,human,Homo sapiens
4,idr0149,Cohesin-mediated DNA loop extrusion resolves s...,protein localization ...,EFO,GO_0008104,Genetic information is stored in linear DNA mo...,Homo sapiens,2023-08-17,37357575,Cohesin-mediated DNA loop extrusion resolves s...,...,EFO EFO,EFO_0003789 EFO_0003969 ...,To assess nuclear Sororin fluorescence in Soro...,Genetic information is stored in linear DNA mo...,"{'CELLULAR_COMPONENT': ['DNA', 'DNA', 'nuclear...",[human],['Homo sapiens'],No,human,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,idr0155,A new method for reliably detecting single flu...,protein localization,EFO,GO_0008104,Super-resolution and single-molecule microscop...,Homo sapiens,2024-03-05,,RASP: Optimal single puncta detection in compl...,...,EFO,EFO_0003969,Tissue sections were incubated with primary an...,Super-resolution and single-molecule microscop...,"{'CELL': ['cells', 'cell', 'cells', 'cellular'...","[human brain, patient]",['Homo sapiens'],No,mouse,Mus musculus
126,idr0044,In Toto Imaging and Reconstruction of Post-Imp...,time-lapse imaging,OMIT,OMIT_0027490,The mouse embryo has long been central to the ...,Mus musculus,2018-11-29,30318151,In Toto Imaging and Reconstruction of Post-Imp...,...,EFO N/A,EFO_0001746 N/A,See attached methods N/A See attached methods ...,The mouse embryo has long been central to the ...,"{'ORGANISM': ['mouse embryo', 'mouse', 'mouse'...","[mouse embryo, mouse, mouse, mouse, E6.5]","['Mus musculus', 'Mus musculus', 'Mus musculus...",No,mouse,Mus

In [36]:
dfs['Species_Scientific_Name_BioBERT'][4] == 'Homo sapiens'

True

In [37]:
for i, item in dfs.iterrows():
    specie_found = item['Specie_found_Scispacy']

    if specie_found.startswith("No"):

        if isinstance(item['Study_Organism'], str) and item['Study_Organism'].strip() != '':
            real_specie = item['Study_Organism'][0]
            bert_specie = item['Species_Scientific_Name_BioBERT'][0]

            if real_specie in bert_specie:

                print('The real species is in the bert')
                dfs.at[i, 'Specie_found_biobert'] = 'Yes'
            else:
                print('The real species is NOT in the response.')
                dfs.at[i, 'Specie_found_biobert'] = 'No'
        else:
            print('No real species to check in the response.')
            dfs.at[i, 'Specie_found_biobert'] = 'No species provided'
  
            

The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real species is in the bert
The real species is in the bert
The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real species is NOT in the response.
The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real species is in the bert
No real species to check in the response.
The real species is in the bert
The real species is in the bert
The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real species is in the bert
The real species is NOT in the response.
The real species is NOT in the response.
The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real spec

In [43]:
dfs['Specie_found_biobert'].value_counts()

Specie_found_biobert
Yes                    61
No                     56
No species provided     6
Name: count, dtype: int64

In [38]:
type(dfs['Specie_found_biobert'][2])

float

In [39]:
dfs

,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Organism,Study_Public_Release_Date,Study_PubMed_ID,Study_Publication_Title,...,Protocol_Type_Term_Accession,Protocol_Description,Description_combined,Entities_SpaCy,Species_SpaCy,Species_Scientific_Name_SciSpacy,Specie_found_Scispacy,Species_BioBERT,Species_Scientific_Name_BioBERT,Specie_found_biobert
0,idr0047,A microscopy data set of discrete spatial and ...,fluorescence in situ hybridization,NCIT,NCIT_C17563,We report a comprehensive single cell dataset ...,Saccharomyces cerevisiae,2019-01-16,31209217,Multiplex RNA single molecule FISH of inducibl...,...,EFO_0003789 EFO_0003969,"CSM, 30C 0.2M NaCl, step TRANS: brightfield of...",We report a comprehensive single cell dataset ...,"{'CELL': ['cell', 'cerevisiae', 'cells', 'cell...",[Excelitas],[],No,yeast,Not found,No
1,idr0038,Ex vivo live cell tracking in kidney organoids...,time-lapse imaging,OMIT,OMIT_0027490,We have adapted the mouse kidney rudiment assa...,Mus musculus,2017-12-04,30048451,Ex vivo live cell tracking in kidney organoids...,...,EFO_0003789,"""Kidneys were isolated from E13.5 embryos afte...",We have adapted the mouse kidney rudiment assa...,"{'ORGANISM': ['mouse kidney rudiment', 'E13.5 ...","[mouse kidney rudiment, E13.5 embryonic kidney...","['Mus musculus', 'Mus musculus', 'Mus musculus...",No,mouse,Mus musculus,Yes
2,idr0146,Time lapse imaging of mesoderm and early cardi...,time-lapse imaging ...,OMIT,OMIT_0027490,Using four-dimensional whole-embryo light shee...,Mus musculus,2023-02-06,36736300,Graded mesoderm assembly governs cell fate and...,...,EFO_0003789 EFO_0003969 ...,see attached DominguezMH Embryo Culture.pdf se...,Using four-dimensional whole-embryo light shee...,"{'ORGANISM': ['murine cardiac', 'mouse', 'E6.5...","[murine cardiac, mouse, E6.5]",['Mus musculus'],Yes,NaN,nan,NaN
3,idr0054,Imaging Mass Cytometry of human tonsil sections,image cytometry,OMIT,OMIT_0019157,We analysed the in situ localisation of immune...,Homo sapiens,2019-03-19,31072818,Human lymphoid organ cDC2 and macrophages play...,...,EFO_0003789 EFO_0003969,Human tonsils were fixed in 4% paraformaldehyd...,We analysed the in situ localisation of immune...,"{'CELL': ['immune cell'], 'ORGANISM': ['human ...","[human tonsils, .Human tonsils, Bovine Serum A...","['Homo sapiens', 'Bos taurus']",No,human,Homo sapiens,Yes
4,idr0149,Cohesin-mediated DNA loop extrusion resolves s...,protein localization ...,EFO,GO_0008104,Genetic information is stored in linear DNA mo...,Homo sapiens,2023-08-17,37357575,Cohesin-mediated DNA loop extrusion resolves s...,...,EFO_0003789 EFO_0003969 ...,To assess nuclear Sororin fluorescence in Soro...,Genetic information is stored in linear DNA mo...,"{'CELLULAR_COMPONENT': ['DNA', 'DNA', 'nuclear...",[human],['Homo sapiens'],No,human,Homo sapiens,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,idr0155,A new method for reliably detecting single flu...,protein localization,EFO,GO_0008104,Super-resolution and single-molecule microscop...,Homo sapiens,2024-03-05,,RASP: Optimal single puncta detection in compl...,...,EFO_0003969,Tissue sections were incubated with primary an...,Super-resolution and single-molecule microscop...,"{'CELL': ['cells', 'cell', 'cells', 'cellular'...","[human brain, patient]",['Homo sapiens'],No,mouse,Mus musculus,No
126,idr0044,In Toto Imaging and Reconstruction of Post-Imp...,time-lapse imaging,OMIT,OMIT_0027490,The mouse embryo has long been central to the ...,Mus musculus,2018-11-29,30318151,In Toto Imaging and Reconstruction of Post-Imp...,...,EFO_0001746 N/A,See attached methods N/A See attached methods ...,The mouse embryo has long been central to the ...,"{'ORGANISM': ['mouse embryo', 'mouse', 'mouse'...","[mouse embryo, mouse, mouse, mouse, E6.5]","['Mus musculus', 'Mus musculus', 'Mus musculus...",No,mouse,Mus musculus,Yes
127,idr0093,High cont

In [44]:
for row, i in dfs.iterrows():
    value_biobert = str(i['Specie_found_biobert'])
    

    if value_biobert.startswith("No"):
    
        message = [
        {"role": "system", "content": i["Description_combined"]},
        {"role": "user", "content": "Can you tell me which specie was used in this study?"},
        ]

        outputs = pipe(message, max_new_tokens = 256)
        response = outputs[0]["generated_text"][-1]["content"]
        dfs.at[row, 'Species_Llama'] = response
        print(f"Processed row {row} for species extraction.")

        if isinstance(i['Study_Organism'], str) and i['Study_Organism'].strip() != '':

            real_specie = i['Study_Organism'][0]

            if real_specie in response:
                print('The real species is in the response.')
                dfs.at[row, 'Species_found_Llama'] = 'Yes'
            else:
                print('The real species is NOT in the response.')
                dfs.at[row, 'Species_found_Llama'] = 'No'
        else:
            print('No real species to check in the response.')
            dfs.at[row, 'Species_found_Llama'] = 'No species provided'

       

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 0 for species extraction.
The real species is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 6 for species extraction.
The real species is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Processed row 124 for species extraction.
The real species is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 125 for species extraction.
The real species is in the response.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed row 128 for species extraction.
The real species is in the response.
Processed row 129 for species extraction.
The real species is in the response.


In [45]:
dfs['Species_found_Llama'].value_counts()

Species_found_Llama
Yes                    39
No                     17
No species provided     6
Name: count, dtype: int64

In [46]:
for row, i in dfs.iterrows():
    value_scipacy = str(i['Specie_found_biobert'])
    value_biobert = str(i['Specie_found_Scispacy'])
    value_lama = str(i['Species_found_Llama'])

    if value_scipacy.startswith("Yes") or value_biobert.startswith("Yes") or value_lama.startswith("Yes"):
        print(f"Row {row}: Species found in one of the methods.")
        dfs.at[row, 'Species_found'] = 'Yes'

    else:
        print(f"Row {row}: Species not found in any method.")
        dfs.at[row, 'Species_found'] = 'No'

Row 0: Species found in one of the methods.
Row 1: Species found in one of the methods.
Row 2: Species found in one of the methods.
Row 3: Species found in one of the methods.
Row 4: Species found in one of the methods.
Row 5: Species found in one of the methods.
Row 6: Species found in one of the methods.
Row 7: Species found in one of the methods.
Row 8: Species found in one of the methods.
Row 9: Species found in one of the methods.
Row 10: Species found in one of the methods.
Row 11: Species found in one of the methods.
Row 12: Species found in one of the methods.
Row 13: Species found in one of the methods.
Row 14: Species not found in any method.
Row 15: Species found in one of the methods.
Row 16: Species found in one of the methods.
Row 17: Species found in one of the methods.
Row 18: Species not found in any method.
Row 19: Species found in one of the methods.
Row 20: Species found in one of the methods.
Row 21: Species found in one of the methods.
Row 22: Species found in one

In [47]:
dfs['Species_found'].value_counts()

Species_found
Yes    107
No      23
Name: count, dtype: int64

In [56]:
for row, i in dfs.iterrows():
    value_scipacy = i['Specie_found_biobert']
    value_biobert = i['Specie_found_Scispacy']
    value_lama = i['Species_found_Llama']

    if value_scipacy.startswith("Yes") :
        print(f"Row {row}: Species found in SciSpacy and Llama, but not in BioBERT.")

Row 1: Species found in SciSpacy and Llama, but not in BioBERT.
Row 3: Species found in SciSpacy and Llama, but not in BioBERT.
Row 4: Species found in SciSpacy and Llama, but not in BioBERT.
Row 5: Species found in SciSpacy and Llama, but not in BioBERT.
Row 7: Species found in SciSpacy and Llama, but not in BioBERT.
Row 8: Species found in SciSpacy and Llama, but not in BioBERT.
Row 11: Species found in SciSpacy and Llama, but not in BioBERT.
Row 12: Species found in SciSpacy and Llama, but not in BioBERT.
Row 15: Species found in SciSpacy and Llama, but not in BioBERT.
Row 16: Species found in SciSpacy and Llama, but not in BioBERT.
Row 17: Species found in SciSpacy and Llama, but not in BioBERT.
Row 19: Species found in SciSpacy and Llama, but not in BioBERT.
Row 20: Species found in SciSpacy and Llama, but not in BioBERT.
Row 22: Species found in SciSpacy and Llama, but not in BioBERT.
Row 23: Species found in SciSpacy and Llama, but not in BioBERT.
Row 24: Species found in SciSpa